# Run 1 (1 year): inject best-fit wavelets, recover them

1. True modulation $P_c(t) = M_c^2(t)$ over the year.
2. Fit wavelet atoms to it: this is the injection.
3. Simulate data with that modulation.
4. Run the RJMCMC sampler.
5. Compare the recovered $P_c(t)$ with the injection.

Settings: [config/wavelet.yaml](../config/wavelet.yaml). `RERUN = True` redoes the fit, the simulation and the sampling;
otherwise the saved files in `outputs/` are loaded.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'simulations' / 'common'))
import injection as I
import fit_reference as FR
import diagnostics as D
from bahamas.wavelet.model import WaveletModulation, DAY
from bahamas.wavelet.simulate import power_modulation
%matplotlib inline

RERUN = False
K_MAX = 12                                   # most atoms per channel the fit tries
CFG = I.load_config(ROOT / 'simulations' / '1_year' / 'config' / 'wavelet.yaml')
OUT = Path(I.out_dir(CFG)) / 'run1'
CH = ['A', 'E']

## 1. True modulation

In [ ]:
t0, T = FR.window(CFG)
t = np.linspace(t0, t0 + T, 3000)
P_true = np.stack([np.asarray(power_modulation(I.sky_params(), t, c)) for c in range(2)])

fig, ax = plt.subplots(figsize=(11, 3))
for i, c in enumerate(CH):
    ax.plot(t / DAY, P_true[i], label=c)
ax.set_xlabel('time [days]'); ax.set_ylabel(r'$M_c^2(t)$'); ax.legend()

## 2. Fit wavelets (the injection)

$\ln P_c(t) = b_c + \sum_k a_k\,e^{-(t-t_k)^2/2\tau_k^2}$, the smallest $K$ with < 1% rms error. Saved with $b_A = 0$
absorbed into the amplitude: `amp_eff` $=$ amp $+ b_A/\ln 10$, `level_E` $= b_E - b_A$.

In [ ]:
ref_path = Path(I.reference_path(CFG))
if RERUN or not ref_path.exists():
    FR.run(config=CFG, k_max=K_MAX)
ref = np.load(ref_path)
wm = WaveletModulation(t0, T, CFG['inference']['wavelet_shape'])

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for i, c in enumerate(CH):
    atoms, b = ref[f'atoms_{c}'], float(ref[f'b_{c}'])
    axes[i].plot(t / DAY, P_true[i], color='0.6', lw=3, label='true')
    axes[i].plot(t / DAY, np.exp(np.asarray(wm.log_modulation(t, atoms, level=b))), 'k--', label=f'fit, K = {len(atoms)}')
    axes[i].set_title(f"{c}: rms {100 * float(ref[f'rms_{c}']):.2f}%"); axes[i].set_xlabel('time [days]'); axes[i].legend()
print(f"amp_eff = {float(ref['amp_eff']):.3f} (true amp {float(ref['amp_true']):.3f}), level_E = {float(ref['level_E']):.3f}")

## 3. Simulate data with the fitted modulation

In [ ]:
data, inj = I.load_or_simulate('wavelet', 'run1', CFG, rerun=RERUN)
like, post = I.build_posterior(data, CFG)
print(f'WDM: {data.Nt} x {data.Nf} pixels; likelihood: {len(like.times)} time blocks x {like.ngroups} frequency groups')

## 4. Sampler

In [ ]:
print('inference settings:', CFG['inference'])
res_path = OUT / 'chains.npz'
if RERUN or not res_path.exists():
    I.run('wavelet', 'run1', CFG)
r = dict(np.load(res_path, allow_pickle=True))
print(f"runtime {float(r['runtime_s']) / 60:.0f} min, acceptance {np.mean(r['acceptance_fraction']):.2f}, "
      f"RJ acceptance {np.mean(r['rj_acceptance_fraction']):.2f}")
D.report(r, post, np.array([60., 180., 300.]) * DAY);

## 5. Recovery

In [ ]:
# number of atoms
fig, axes = plt.subplots(1, 2, figsize=(12, 2.8))
for ax, c in zip(axes, CH):
    K, p, K_true = r[f'mod_{c}_bayes_K'], r[f'mod_{c}_bayes_posterior'], len(inj.atoms[c])
    ax.bar(K, p, color=['C1' if k == K_true else 'C0' for k in K])
    ax.set_title(f'{c}: p(K), true K = {K_true} (orange)'); ax.set_xlabel('K')

In [ ]:
# modulation: 10^(amp - amp_inj) P_c(t), 90% band vs the injection
td = r['modulation_times'] / DAY
q = np.percentile(r['modulation_draws'], [5, 50, 95], axis=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
for i, c in enumerate(CH):
    axes[i].fill_between(td, q[0, i], q[2, i], alpha=0.4, label='90%')
    axes[i].plot(td, q[1, i], label='median')
    axes[i].plot(td, r['P_true'][i], 'k--', label='injected')
    axes[i].set_title(c); axes[i].set_xlabel('time [days]'); axes[i].legend()
    inside = np.mean((q[0, i] <= r['P_true'][i]) & (r['P_true'][i] <= q[2, i]))
    print(f'{c}: injection inside the 90% band at {100 * inside:.0f}% of time pixels')
fig.savefig(OUT / 'modulation_recovery.png', dpi=150)

In [ ]:
# spectral parameters and level_E
import corner
names = [str(n).split(':')[-1] for n in r['psd_names']]
corner.corner(r['psd_chain'], labels=names, truths=r['psd_injected'], show_titles=True, title_fmt='.3f')
plt.savefig(OUT / 'corner_psd.png', dpi=110);